# Construcción del modelo de PuntoBot

Desarrollo del clasificador seleccionado en la etapa 4. La entrada es texto; no se traslada ResNet50 del ejemplo dermatológico a consultas de farmacia.

En Colab, autoriza un secreto llamado `GITHUB_TOKEN` con lectura de este repositorio privado. En tu PC, instala `requirements-keras.txt` y abre el notebook desde el repositorio. Nunca pegues el token en una celda. Los datos de demostración son sintéticos; `test.csv` se mantiene sellado.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, io, urllib.request, subprocess

# En Colab: abre el notebook desde GitHub y autoriza el secreto GITHUB_TOKEN.
# El token se usa solo en una cabecera HTTP y nunca se imprime ni se guarda.
if Path('/content').exists() and not Path('config.yaml').exists():
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    request = urllib.request.Request(
        'https://api.github.com/repos/rebirth18-09/puntobot-ia/zipball/main',
        headers={'Authorization': 'Bearer ' + token, 'Accept': 'application/vnd.github+json'})
    with urllib.request.urlopen(request) as response:
        contenido = response.read()
    del token, request
    destino = Path('/content/puntobot-ia')
    destino.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(contenido)) as archivo:
        for item in archivo.infolist():
            target = (destino / item.filename).resolve()
            if not target.is_relative_to(destino.resolve()):
                raise ValueError('Ruta inválida en el ZIP')
        archivo.extractall(destino)
    os.chdir(next(p for p in destino.iterdir() if (p / 'config.yaml').exists()))
RAIZ = Path.cwd()
if not (RAIZ / 'config.yaml').exists() and (RAIZ.parent / 'config.yaml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / 'config.yaml').exists(), 'Ejecuta el notebook desde la carpeta del repositorio'
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
if Path('/content').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-keras.txt'], check=True)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import tensorflow as tf
import keras
import numpy as np
import yaml
try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass
config = yaml.safe_load((RAIZ / 'config.yaml').read_text('utf-8'))
keras.utils.set_random_seed(config['entrenamiento']['semilla'])
print('TensorFlow', tf.__version__, 'Keras', keras.__version__)
print('GPU', tf.config.list_physical_devices('GPU'))

## Construir el pipeline sin entrenar

Dos representaciones TF-IDF: palabras de uno a dos términos y caracteres de tres a cinco. La regresión logística recibe sus características, aplica pesos balanceados y produce probabilidades de cuatro intenciones.

In [ ]:
from src.models.modelo import crear_modelo
modelo = crear_modelo(config)
assert not hasattr(modelo.named_steps['clasificador'], 'coef_')
print(modelo)
print('Clases del contrato:', config['clases'])
print('Modelo construido sin ajustar.')

## Contrato de inferencia

`src/inference/predictor.py` devuelve versión, probabilidades, producto propuesto y candidatos. No aplica los umbrales del backend ni devuelve existencias, precios o ventas. El endpoint HTTP se implementará al integrar el sistema.

Se conserva la salida `.joblib` acordada en la arquitectura: el vectorizador y el clasificador se guardan juntos. El catálogo de esta demostración se obtiene únicamente de train; para operar se requiere el catálogo real de la farmacia.

## Decisiones de desarrollo

No se usa StandardScaler: TF-IDF ya normaliza los vectores. No se aplica aumentación aleatoria al texto porque podría cambiar la intención o el producto. El conteo de etiquetas y los pesos se calculan con train. Los hiperparámetros se leen desde `config.yaml`. La etapa 7 realizará entrenamiento, selección y evaluación final con el corpus real.

## Comprobar dimensiones, clases y pesos

El objeto anterior sigue sin entrenar. Este segundo objeto se ajusta solo como prueba de cordura, usando train; se descarta al cerrar el notebook. No selecciona hiperparámetros ni utiliza las etiquetas de prueba. Comprueba el equivalente de la forma de salida y el conteo de parámetros de la sesión 23.


In [ ]:
from src.data.division import cargar_particiones
from src.models.pesos import calcular_pesos
train, val, auditoria = cargar_particiones(config, RAIZ)
texto, etiqueta = config['datos']['columna_texto'], config['datos']['columna_clase']
pesos = calcular_pesos(train[etiqueta].tolist(), config['clases'])
modelo_cordura = crear_modelo(config).fit(train[texto], train[etiqueta])
X = modelo_cordura.named_steps['tfidf'].transform(train[texto])
clasificador = modelo_cordura.named_steps['clasificador']
k = len(config['clases'])
assert clasificador.coef_.shape == (k, X.shape[1])
parametros = clasificador.coef_.size + clasificador.intercept_.size
assert parametros == (X.shape[1] + 1) * k
salida = modelo_cordura.predict_proba(val[texto].iloc[:4])
assert salida.shape == (4, k)
assert np.isfinite(salida).all() and ((salida >= 0) & (salida <= 1)).all()
assert np.allclose(salida.sum(axis=1), 1)
assert set(modelo_cordura.classes_) == set(config['clases'])
assert not hasattr(modelo.named_steps['clasificador'], 'coef_')
resumen = {'salida': list(salida.shape), 'caracteristicas': X.shape[1],
    'coeficientes': list(clasificador.coef_.shape), 'parametros': int(parametros),
    'pesos_solo_train': pesos, 'modelo_original_sin_ajustar': True,
    'probabilidades_validas': True, 'prueba_sin_evaluar': True}
print(json.dumps(resumen, ensure_ascii=False, indent=2))
print('Salida correcta; el clasificador construido permanece sin ajustar.')


## Documentación y continuidad

- [Diseño y justificación](../docs/06-desarrollo-modelo/diseno-del-modelo.md).
- [ADR-003](../docs/06-desarrollo-modelo/decisiones/ADR-003-tfidf-regresion-logistica.md).

En regresión logística no hay una base preentrenada que congelar. El conteo de coeficientes se obtiene después del ajuste de cordura; en Keras se obtiene desde la construcción. Esta diferencia se documenta como adaptación a PuntoBot.
